# RI-JK RHF Hessian：交换分解

In [1]:
from pyscf import gto, scf, lib
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_K20 = np.load("nh3_r_hf_decomp.npz")["de_K20"]
de_K11 = np.load("nh3_r_hf_decomp.npz")["de_K11"]
de_K02 = np.load("nh3_r_hf_decomp.npz")["de_K02"]

In [6]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)

# 详细分解

详细分解的大体原则：

- 程序越简单越好。完全不考虑效率。
- 我们当前不考虑所有电子积分的对称性。所有电子积分直接存到对应的变量里。
- 所有运算使用 einsum (如果要在外部 Python 程序执行，记得要加 optimize=True，即使效率在这里不关键但不加该选项的执行时间会非常大；本 notebook 是因为前面有了 `functools.partial` 重定义了 np.einsum 的默认参数为 `optimize="greedy"`，所以可以不用再加)。
- 首先得到一个关于基组 / 辅助基的、与原子无关的贡献矩阵 (例如 `dbas_J_20_ip1_contrib`)；这个矩阵应该需要足够小 (一般是 3 x 3 x nao/naux x nao/naux)，这个储存大小比较小，方便后续处理。
- 然后通过一个双重循环 (A, B) 来将基组 / 辅助基的贡献矩阵转换为原子贡献矩阵 (例如 `de_J_20_ip1_contrib[A, B]`)。留意有一些贡献项是只对单个原子 (A) 循环的 (例如 `de_J_20_ipip1_contrib[A, A]`)。这里也同时处理一些系数缩放 (譬如 `de_J20_ip1_contrib` 的 4 倍)。
- 最后将结果拼起来，用 `np.allclose` 检查是否相等。

- 对于 K (交换积分) 的贡献，我们经常需要使用 `occ_coeff` 与 `occ_occupation` 变量以对其中一个原子轨道，缩并到分子轨道。

In [7]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])
int3c2e_ipip1 = _int3c_wrapper(mol, aux, "int3c2e_ipip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipvip1 = _int3c_wrapper(mol, aux, "int3c2e_ipvip1", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ip1ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip1ip2", "s1")().reshape([3, 3, nao, nao, naux])
int3c2e_ipip2 = _int3c_wrapper(mol, aux, "int3c2e_ipip2", "s1")().reshape([3, 3, nao, nao, naux])
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int2c2e_ipip1 = aux.intor("int2c2e_ipip1").reshape([3, 3, naux, naux])
int2c2e_ip1ip2 = aux.intor("int2c2e_ip1ip2").reshape([3, 3, naux, naux])

### K (basis_2nd)

In [8]:
# (10|0)(0|10), part a
dbas_K20_1a = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_1a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_1a[A, B] += 2 * np.einsum("tsuk -> ts", dbas_K20_1a[:, :, p0A:p1A, p0B:p1B])

In [9]:
# (10|0)(0|10), part b
dbas_K20_1b = np.einsum("tuvP, PQ, sklQ, ui, vj, kj, li -> tsuk", int3c2e_ip1, int2c2e_inv, int3c2e_ip1, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_1b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_1b[A, B] += 2 * np.einsum("tsuk -> ts", dbas_K20_1b[:, :, p0A:p1A, p0B:p1B])

In [10]:
# (11|0)(0|00)
dbas_K20_2 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsuv", int3c2e_ipvip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(aoslices):
        de_K20_2[A, B] += 2 * np.einsum("tsuv -> ts", dbas_K20_2[:, :, p0A:p1A, p0B:p1B])

In [11]:
# (20|0)(0|00)
dbas_K20_3 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsuv", int3c2e_ipip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K20_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    de_K20_3[A, A] += 2 * np.einsum("tsuv -> ts", dbas_K20_3[:, :, p0A:p1A])

In [12]:
de_K20_recap = de_K20_1a + de_K20_1b + de_K20_2 + de_K20_3
assert np.allclose(de_K20_recap, de_K20)

### K (basis_1st_aux_1st)

In [13]:
# (10|1)(0|0)(0|00)
dbas_K11_1 = np.einsum("tsuvP, PQ, klQ, vi, li, uj, kj -> tsuP", int3c2e_ip1ip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_1[A, B] += 2 * np.einsum("tsuP -> ts", dbas_K11_1[:, :, p0A:p1A, p0B:p1B])
de_K11_1 += de_K11_1.transpose(1, 0, 3, 2)

In [14]:
# (10|0)(0|1)(0|00)
dbas_K11_2 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsuR", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_2[A, B] += 2 * np.einsum("tsuR -> ts", dbas_K11_2[:, :, p0A:p1A, p0B:p1B])
de_K11_2 += de_K11_2.transpose(1, 0, 3, 2)

In [15]:
# (10|0)(1|0)(0|00)
dbas_K11_3 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsuQ", int3c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_3 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_3[A, B] += -2 * np.einsum("tsuQ -> ts", dbas_K11_3[:, :, p0A:p1A, p0B:p1B])
de_K11_3 += de_K11_3.transpose(1, 0, 3, 2)

In [16]:
# (10|0)(0|0)(1|00)
dbas_K11_4 = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsuQ", int3c2e_ip1, int2c2e_inv, int3c2e_ip2, mocc_2, mocc_2, mocc_2, mocc_2)
de_K11_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(aoslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K11_4[A, B] += 2 * np.einsum("tsuQ -> ts", dbas_K11_4[:, :, p0A:p1A, p0B:p1B])
de_K11_4 += de_K11_4.transpose(1, 0, 3, 2)

In [17]:
de_K11_recap = de_K11_1 + de_K11_2 + de_K11_3 + de_K11_4
assert np.allclose(de_K11_recap, de_K11)

### K (aux_2nd)

In [18]:
# (00|2)(0|00)
dbas_K02_1 = np.einsum("tsuvP, PQ, klQ, ui, vj, ki, lj -> tsP", int3c2e_ipip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_1 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_K02_1[A, A] += np.einsum("tsP -> ts", dbas_K02_1[:, :, p0A:p1A])

In [19]:
# (00|0)(2|0)(0|00)
dbas_K02_2 = np.einsum("uvP, PQ, tsQR, RS, klS, ui, vj, ki, lj -> tsQ", int3c2e, int2c2e_inv, int2c2e_ipip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_2 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    de_K02_2[A, A] += -1 * np.einsum("tsQ -> ts", dbas_K02_2[:, :, p0A:p1A])
de_K02_2 = de_K02_2

In [20]:
# (00|0)(1|1)(0|00)
dbas_K02_3a = np.einsum("uvP, PQ, tsQR, RS, klS, ui, vj, ki, lj -> tsQR", int3c2e, int2c2e_inv, int2c2e_ip1ip2, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_3a = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_3a[A, B] += -0.5 * np.einsum("tsQR -> ts", dbas_K02_3a[:, :, p0A:p1A, p0B:p1B])
de_K02_3a += de_K02_3a.transpose(1, 0, 3, 2)

In [21]:
# (00|0)(1|0)(0|1)(0|00)
dbas_K02_3b = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, ui, vj, ki, lj -> tsQT", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_3b = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_3b[A, B] += -0.5 * np.einsum("tsQT -> ts", dbas_K02_3b[:, :, p0A:p1A, p0B:p1B])
de_K02_3b += de_K02_3b.transpose(1, 0, 3, 2)

In [22]:
# (00|1)(1|0)(0|00)
dbas_K02_4 = np.einsum("tuvP, PQ, sQR, RS, klS, ui, vj, ki, lj -> tsPQ", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_4 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_4[A, B] += -1 * np.einsum("tsPQ -> ts", dbas_K02_4[:, :, p0A:p1A, p0B:p1B])
de_K02_4 += de_K02_4.transpose(1, 0, 3, 2)

In [23]:
# (00|1)(1|00)
dbas_K02_5 = np.einsum("tuvP, PQ, sklQ, ui, vj, ki, lj -> tsPQ", int3c2e_ip2, int2c2e_inv, int3c2e_ip2, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_5 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_5[A, B] += 0.5 * np.einsum("tsPQ -> ts", dbas_K02_5[:, :, p0A:p1A, p0B:p1B])
de_K02_5 += de_K02_5.transpose(1, 0, 3, 2)

In [24]:
# (00|0)(0|1)(1|0)(0|00)
dbas_K02_6 = np.einsum("uvP, PQ, tRQ, RS, sST, TU, klU, ui, vj, ki, lj -> tsRS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_6 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_6[A, B] += 0.5 * np.einsum("tsRS -> ts", dbas_K02_6[:, :, p0A:p1A, p0B:p1B])
de_K02_6 += de_K02_6.transpose(1, 0, 3, 2)

In [25]:
# (00|1)(0|1)(0|00)
dbas_K02_7 = np.einsum("tuvP, PQ, sRQ, RS, klS, ui, vj, ki, lj -> tsPR", int3c2e_ip2, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_7 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_7[A, B] += -1 * np.einsum("tsPR -> ts", dbas_K02_7[:, :, p0A:p1A, p0B:p1B])
de_K02_7 += de_K02_7.transpose(1, 0, 3, 2)

In [26]:
# (00|0)(1|0)(1|0)(0|00)
dbas_K02_8 = np.einsum("uvP, PQ, tQR, RS, sST, TU, klU, ui, vj, ki, lj -> tsQS", int3c2e, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int2c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2, mocc_2, mocc_2)
de_K02_8 = np.zeros((natm, natm, 3, 3))
for A, (_, _, p0A, p1A) in enumerate(auxslices):
    for B, (_, _, p0B, p1B) in enumerate(auxslices):
        de_K02_8[A, B] += 1 * np.einsum("tsQS -> ts", dbas_K02_8[:, :, p0A:p1A, p0B:p1B])
de_K02_8 += de_K02_8.transpose(1, 0, 3, 2)

In [27]:
de_K02_recap = de_K02_1 + de_K02_2 + de_K02_3a + de_K02_3b + de_K02_4 + de_K02_5 + de_K02_6 + de_K02_7 + de_K02_8
assert np.allclose(de_K02_recap, de_K02, atol=1e-5, rtol=1e-4)

## 存储到文件

In [28]:
dat = dict(np.load("nh3_r_hf_decomp.npz"))
dat.update({
    # de_K20
    "de_K20_1a": de_K20_1a,
    "de_K20_1b": de_K20_1b,
    "de_K20_2": de_K20_2,
    "de_K20_3": de_K20_3,
    # de_K11
    "de_K11_1": de_K11_1,
    "de_K11_2": de_K11_2,
    "de_K11_3": de_K11_3,
    "de_K11_4": de_K11_4,
    # de_K02
    "de_K02_1": de_K02_1,
    "de_K02_2": de_K02_2,
    "de_K02_3a": de_K02_3a,
    "de_K02_3b": de_K02_3b,
    "de_K02_4": de_K02_4,
    "de_K02_5": de_K02_5,
    "de_K02_6": de_K02_6,
    "de_K02_7": de_K02_7,
    "de_K02_8": de_K02_8,
})
np.savez("nh3_r_hf_decomp.npz", **dat)